### Calculating the QM solution of the reaction between $OH^-$ and $CH_3Cl$

In [1]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyscf import gto, scf
from pyscf.geomopt.geometric_solver import optimize
from pyscf.qmmm import mm_charge
from scipy.optimize import minimize
from scipy.spatial.transform import Rotation
import pickle
from pyscf.grad import rhf as rhf_grad

In [2]:
au_to_kJ_conversion = 2625.49962
R = 8.314462618  # J/(mol K)
SCF_CONV_TOL = 1e-6

In [3]:
def qm_water_lj_energy(qm_coords_A, water_O_coords_A, qm_atom_types):
    water_O = np.asarray(water_O_coords_A[0])

    E_LJ = 0.0

    for i, atom_type in enumerate(qm_atom_types):

        # Skip atoms for which we have no LJ parameters
        if atom_type not in qm_lj:
            continue

        sigma = qm_lj[atom_type]["sigma_A"]
        epsilon = qm_lj[atom_type]["epsilon_kJmol"]

        r = np.linalg.norm(
            qm_coords_A[i] - water_O
        )

        E_LJ += lj_energy(
            r,
            sigma,
            epsilon
        )

    return E_LJ

def water_from_variables(x):
    """
    x[:3] = O position in Angstrom
    x[3:6] = rotation vector in radians

    Returns:
        O, H1, H2, M
    """

    translation = np.asarray(x[:3])
    rotation = Rotation.from_rotvec(x[3:6])

    O  = translation + rotation.apply(water_O_local)
    H1 = translation + rotation.apply(water_H1_local)
    H2 = translation + rotation.apply(water_H2_local)
    M  = translation + rotation.apply(water_M_local)

    return O, H1, H2, M

def total_qm_water_energy(x, mol, qm_atom_types, energy_qm):

    # ------------------------------------------------
    # Safety check on water translation
    # ------------------------------------------------

    if np.max(np.abs(x[:3])) > 10.0:
        return 1.0e10

    # Build water
    O, H1, H2, M = water_from_variables(x)

    # ------------------------------------------------
    # TIP4P-D electrostatic sites
    # ------------------------------------------------

    mm_coords = np.array([
        H1,
        H2,
        M
    ])

    mm_charges = np.array([
        0.58,
        0.58,
        -1.16
    ])

    # ------------------------------------------------
    # QM/MM electrostatics
    # ------------------------------------------------

    mf_qmmm = mm_charge(
        scf.RHF(mol),
        mm_coords,
        mm_charges,
        unit="Angstrom"
    )
    mf_qmmm.conv_tol = SCF_CONV_TOL
    energy_qmmm = mf_qmmm.kernel()

    # ------------------------------------------------
    # QM-water LJ interaction
    # ------------------------------------------------

    E_LJ = qm_water_lj_energy(
        mol.atom_coords(unit="Angstrom"),
        np.array([O]),
        qm_atom_types
    )

    # ------------------------------------------------
    # Electrostatic interaction
    # ------------------------------------------------

    E_electrostatic = (
        energy_qmmm - energy_qm
    ) * au_to_kJ_conversion

    return E_electrostatic + E_LJ

def lj_energy(r_A, sigma_A, epsilon_kJmol):
    """
    Lennard-Jones 12-6 energy.

    Parameters
    ----------
    r_A : float
        Distance in Angstrom
    sigma_A : float
        LJ sigma in Angstrom
    epsilon_kJmol : float
        LJ epsilon in kJ/mol

    Returns
    -------
    float
        LJ energy in kJ/mol
    """
    sr6 = (sigma_A / r_A)**6
    return 4.0 * epsilon_kJmol * (sr6**2 - sr6)

def build_molecule(coords):

    atom_string = ""

    for symbol, coord in zip(symbols, coords):

        x, y, z = coord

        atom_string += (
            f"{symbol} "
            f"{x:.10f} "
            f"{y:.10f} "
            f"{z:.10f}\n"
        )

    return gto.M(
        atom=atom_string,
        basis="6-31G",
        charge=-1,
        spin=0,
        unit="Angstrom"
    )

def move_oh_to_distance(coords, target_distance):

    coords = coords.copy()

    O = 0
    H_oh = 1
    C = 2

    carbon = coords[C]
    oxygen = coords[O]

    # Current direction from C toward O
    direction = oxygen - carbon
    direction /= np.linalg.norm(direction)

    # Preserve the O-H vector
    oh_vector = coords[H_oh] - coords[O]

    # Place O at the desired C-O distance
    new_oxygen = carbon + direction * target_distance

    # Move H together with O
    new_hydrogen = new_oxygen + oh_vector

    coords[O] = new_oxygen
    coords[H_oh] = new_hydrogen

    return coords

def write_constraint(distance):

    with open("constraints.txt", "w") as f:

        f.write("$set\n")
        f.write(
            f"distance 1 3 {distance:.8f}\n"
        )

def optimize_at_distance(mol, distance, maxsteps=25):

    # Create the constraint file
    write_constraint(distance)

    # Build RHF + implicit solvent
    mf = make_scf(mol)

    # Optimize geometry while keeping C-O fixed
    mol_opt = optimize(
        mf,
        constraints="constraints.txt",
        maxsteps=maxsteps
    )

    # Recalculate final energy using the optimized geometry
    mf_final = make_scf(mol_opt)
    energy = mf_final.kernel()

    # Return the optimized geometry and energy
    return mol_opt, energy

def make_scf(mol):

    mf = scf.RHF(mol)
    mf.conv_tol = SCF_CONV_TOL
    return mf

def total_qmmm_lj_energy(coords_A, water_x):

    # Build QM molecule
    mol = build_molecule(coords_A)

    # Build TIP4P-D water
    O, H1, H2, M = water_from_variables(water_x)

    mm_coords = np.array([
        H1,
        H2,
        M
    ])

    mm_charges = np.array([
        0.58,
        0.58,
        -1.16
    ])

    # ------------------------------------------------
    # Bare QM energy
    # ------------------------------------------------

    mf_qm = scf.RHF(mol)
    mf_qm.conv_tol = SCF_CONV_TOL
    E_qm = mf_qm.kernel()

    # ------------------------------------------------
    # QM/MM energy
    # ------------------------------------------------

    mf_qmmm = mm_charge(
        scf.RHF(mol),
        mm_coords,
        mm_charges,
        unit="Angstrom"
    )

    mf_qmmm.conv_tol = SCF_CONV_TOL
    E_qmmm = mf_qmmm.kernel()

    # ------------------------------------------------
    # QM/MM electrostatic interaction
    # ------------------------------------------------

    E_electrostatic = (
        E_qmmm - E_qm
    ) * au_to_kJ_conversion

    # ------------------------------------------------
    # QM-water LJ
    # ------------------------------------------------

    E_LJ = qm_water_lj_energy(
        coords_A,
        np.array([O]),
        qm_atom_types
    )

    # ------------------------------------------------
    # Total interaction energy
    # ------------------------------------------------

    E_total = E_electrostatic + E_LJ

    return E_total
def optimize_qmmm_at_distance(
    mol,
    distance,
    water_x,
    maxiter=25
):

    coords0 = mol.atom_coords(
        unit="Angstrom"
    )

    # ------------------------------------------------
    # C-O distance constraint
    # ------------------------------------------------

    def co_distance(coords_flat):

        coords = coords_flat.reshape((-1, 3))

        C = coords[2]
        O = coords[0]

        return np.linalg.norm(O - C)

    constraint = {
        "type": "eq",
        "fun": lambda x: co_distance(x) - distance
    }

    # ------------------------------------------------
    # QM/MM optimization
    # ------------------------------------------------

    result = minimize(
        lambda x: qmmm_objective(x, water_x),
        coords0.reshape(-1),
        jac=True,
        method="SLSQP",
        constraints=[constraint],
        options={
            "maxiter": maxiter,
            "ftol": 1e-5,
            "disp": True
        }
    )

    # ------------------------------------------------
    # Build optimized molecule
    # ------------------------------------------------

    coords_opt = result.x.reshape((-1, 3))

    mol_opt = build_molecule(coords_opt)

    # ------------------------------------------------
    # Final energies
    # ------------------------------------------------

    mf_qm = make_scf(mol_opt)
    energy_qm = mf_qm.kernel()

    energy_qmmm, _ = qmmm_gradient(
        mol_opt,
        water_x
    )

    return (
        mol_opt,
        energy_qm,
        energy_qmmm,
        result
    )
    
    
def qm_water_lj_energy_gradient(
    qm_coords_A,
    water_O_coords_A,
    qm_atom_types
):
    """
    QM-water LJ energy and gradient.

    Returns
    -------
    E_LJ : float
        LJ energy in kJ/mol

    grad : ndarray, shape (N,3)
        Gradient dE/d(R_QM) in kJ/mol/Angstrom
    """

    water_O = np.asarray(water_O_coords_A[0])

    E_LJ = 0.0
    grad = np.zeros_like(qm_coords_A, dtype=float)

    for i, atom_type in enumerate(qm_atom_types):

        if atom_type not in qm_lj:
            continue

        sigma = qm_lj[atom_type]["sigma_A"]
        epsilon = qm_lj[atom_type]["epsilon_kJmol"]

        # Vector from water O -> QM atom
        dr = qm_coords_A[i] - water_O

        r = np.linalg.norm(dr)

        sr6 = (sigma / r)**6

        # LJ energy
        E_i = 4.0 * epsilon * (sr6**2 - sr6)

        E_LJ += E_i

        # dE/dr
        dE_dr = (
            24.0 * epsilon / r
            * (sr6 - 2.0 * sr6**2)
        )

        # dE/dR_vector
        grad[i] += dE_dr * dr / r

    return E_LJ, grad


class QMMM_LJ_Gradients:

    def __init__(
        self,
        mf,
        water_O_A,
        qm_atom_types,
        mm_coords,
        mm_charges
    ):

        self.mf = mf
        self.mol = mf.mol

        self.water_O_A = np.asarray(water_O_A)
        self.qm_atom_types = qm_atom_types

        self.mm_coords = np.asarray(mm_coords)
        self.mm_charges = np.asarray(mm_charges)

        # Attributes expected by PySCF/geomeTRIC
        self.verbose = mf.verbose
        self.stdout = mf.stdout
        self.converged = True

    def nuc_grad_method(self):
        return self

    def as_scanner(self):
        return self

    def __call__(self, mol):

        # --------------------------------------------
        # QM/MM SCF
        # --------------------------------------------

        mf_qmmm = mm_charge(
            scf.RHF(mol),
            self.mm_coords,
            self.mm_charges,
            unit="Angstrom"
        )

        mf_qmmm.conv_tol = SCF_CONV_TOL

        E_qmmm = mf_qmmm.kernel()

        # --------------------------------------------
        # QM/MM nuclear gradient
        # --------------------------------------------

        grad_qmmm = (
            mf_qmmm
            .nuc_grad_method()
            .kernel()
        )

        # --------------------------------------------
        # LJ energy + gradient
        # --------------------------------------------

        qm_coords_A = mol.atom_coords(
            unit="Angstrom"
        )

        E_LJ, grad_LJ = qm_water_lj_energy_gradient(
            qm_coords_A,
            self.water_O_A,
            self.qm_atom_types
        )

        # --------------------------------------------
        # Convert LJ gradient:
        #
        # kJ/mol/Angstrom
        #        ->
        # Hartree/Bohr
        # --------------------------------------------

        grad_LJ_Ha_Bohr = (
            grad_LJ
            / au_to_kJ_conversion
            / 1.889726125
        )

        # --------------------------------------------
        # Combined energy
        # --------------------------------------------

        E_total = (
            E_qmmm
            + E_LJ / au_to_kJ_conversion
        )

        # --------------------------------------------
        # Combined gradient
        # --------------------------------------------

        grad_total = (
            grad_qmmm
            + grad_LJ_Ha_Bohr
        )

        return E_total, grad_total

def optimize_qmmm_lj_at_distance(
    mol,
    distance,
    water_x,
    maxsteps=25
):

    # ------------------------------------------------
    # 1. Write the C-O constraint
    # ------------------------------------------------

    write_constraint(distance)

    # ------------------------------------------------
    # 2. Build the current TIP4P-D water
    # ------------------------------------------------

    O, H1, H2, M = water_from_variables(water_x)

    mm_coords = np.array([
        H1,
        H2,
        M
    ])

    mm_charges = np.array([
        0.58,
        0.58,
        -1.16
    ])

    # ------------------------------------------------
    # 3. Create a QM/MM SCF object
    # ------------------------------------------------

    mf = mm_charge(
        scf.RHF(mol),
        mm_coords,
        mm_charges,
        unit="Angstrom"
    )

    mf.conv_tol = SCF_CONV_TOL

    # ------------------------------------------------
    # 4. Create our combined QM/MM + LJ gradient
    # ------------------------------------------------

    grad = QMMM_LJ_Gradients(
        mf,
        O,
        qm_atom_types,
        mm_coords,
        mm_charges
    )

    # ------------------------------------------------
    # 5. Geometry optimization
    # ------------------------------------------------

    mol_opt = optimize(
        grad,
        constraints="constraints.txt",
        maxsteps=maxsteps
    )

    # ------------------------------------------------
    # 6. Bare QM energy at optimized geometry
    # ------------------------------------------------

    mf_qm = scf.RHF(mol_opt)
    mf_qm.conv_tol = SCF_CONV_TOL

    energy_qm = mf_qm.kernel()

    # ------------------------------------------------
    # 7. Final QM/MM energy
    # ------------------------------------------------

    mf_qmmm_final = mm_charge(
        scf.RHF(mol_opt),
        mm_coords,
        mm_charges,
        unit="Angstrom"
    )

    mf_qmmm_final.conv_tol = SCF_CONV_TOL

    energy_qmmm = mf_qmmm_final.kernel()

    # ------------------------------------------------
    # 8. Final LJ energy
    # ------------------------------------------------

    E_LJ = qm_water_lj_energy(
        mol_opt.atom_coords(unit="Angstrom"),
        np.array([O]),
        qm_atom_types
    )

    # ------------------------------------------------
    # 9. Return everything
    # ------------------------------------------------

    return (
        mol_opt,
        energy_qm,
        energy_qmmm,
        E_LJ,
        None
    )

def qmmm_gradient(mol, water_x):

    O, H1, H2, M = water_from_variables(water_x)

    mm_coords = np.array([
        H1,
        H2,
        M
    ])

    mm_charges = np.array([
        0.58,
        0.58,
        -1.16
    ])

    mf_qmmm = mm_charge(
        scf.RHF(mol),
        mm_coords,
        mm_charges,
        unit="Angstrom"
    )

    mf_qmmm.conv_tol = SCF_CONV_TOL

    energy = mf_qmmm.kernel()

    grad = mf_qmmm.nuc_grad_method().kernel()

    return energy, grad

def qmmm_objective(coords_flat, water_x):
    """
    QM/MM energy and gradient for scipy.optimize.

    coords_flat: QM coordinates flattened, in Angstrom
    """

    coords_A = coords_flat.reshape((-1, 3))

    mol = build_molecule(coords_A)

    energy, grad = qmmm_gradient(
        mol,
        water_x
    )

    # Convert gradient from Hartree/Bohr
    # to Hartree/Angstrom because our optimization
    # variables are in Angstrom.
    grad_A = grad / 1.889726125

    return energy, grad_A.reshape(-1)

def qmmm_lj_objective(coords_flat, water_x):
    """
    QM/MM + QM-water LJ energy and gradient.

    coords_flat:
        QM coordinates, flattened, in Angstrom.

    water_x:
        TIP4P-D water position/orientation.

    Returns
    -------
    energy : float
        QM/MM + LJ energy in Hartree.

    gradient : ndarray
        Total gradient in Hartree/Angstrom.
    """

    # ------------------------------------------------
    # Build QM molecule
    # ------------------------------------------------

    coords_A = coords_flat.reshape((-1, 3))

    mol = build_molecule(coords_A)

    # ------------------------------------------------
    # QM/MM energy and gradient
    # ------------------------------------------------

    E_qmmm, grad_qmmm = qmmm_gradient(
        mol,
        water_x
    )

    # grad_qmmm is Hartree/Bohr
    # Convert to Hartree/Angstrom
    grad_qmmm_A = (
        grad_qmmm / 1.889726125
    )

    # ------------------------------------------------
    # TIP4P-D water
    # ------------------------------------------------

    O, H1, H2, M = water_from_variables(water_x)

    # ------------------------------------------------
    # LJ energy and gradient
    # ------------------------------------------------

    E_LJ, grad_LJ = qm_water_lj_energy_gradient(
        coords_A,
        np.array([O]),
        qm_atom_types
    )

    # E_LJ is kJ/mol
    # grad_LJ is kJ/mol/Angstrom

    # Convert LJ energy to Hartree
    E_LJ_Hartree = (
        E_LJ / au_to_kJ_conversion
    )

    # Convert LJ gradient to Hartree/Angstrom
    grad_LJ_Hartree_A = (
        grad_LJ / au_to_kJ_conversion
    )

    # ------------------------------------------------
    # Total
    # ------------------------------------------------

    E_total = (
        E_qmmm
        + E_LJ_Hartree
    )

    grad_total = (
        grad_qmmm_A
        + grad_LJ_Hartree_A
    )

    return E_total, grad_total.reshape(-1)

def qmmm_lj_energy_gradient(coords_flat, water_x):
    """
    QM/MM + LJ energy and gradient for fixed TIP4P-D water.

    Parameters
    ----------
    coords_flat : array, shape (21,)
        QM Cartesian coordinates in Angstrom.
    water_x : array, shape (6,)
        Fixed TIP4P-D water position/orientation.

    Returns
    -------
    energy : float
        Total QM/MM + LJ energy in kJ/mol.

    gradient : array, shape (21,)
        Total gradient in kJ/mol/Angstrom.
    """

    # ------------------------------------------------------------
    # QM coordinates
    # ------------------------------------------------------------

    coords_A = np.asarray(coords_flat).reshape(-1, 3)

    mol = build_molecule(coords_A)

    # ------------------------------------------------------------
    # Fixed TIP4P-D water
    # ------------------------------------------------------------

    O, H1, H2, M = water_from_variables(water_x)

    mm_coords = np.array([
        H1,
        H2,
        M
    ])

    mm_charges = np.array([
        0.58,
        0.58,
        -1.16
    ])

    # ------------------------------------------------------------
    # QM/MM electrostatics
    # ------------------------------------------------------------

    mf_qmmm = mm_charge(
        scf.RHF(mol),
        mm_coords,
        mm_charges,
        unit="Angstrom"
    )

    mf_qmmm.conv_tol = SCF_CONV_TOL

    E_qmmm_Ha = mf_qmmm.kernel()

    grad_qmmm_Ha_Bohr = (
        mf_qmmm.nuc_grad_method().kernel()
    )

    # Convert QM/MM gradient:
    #
    # Hartree/Bohr -> kJ/mol/Angstrom
    #
    grad_qmmm = (
        grad_qmmm_Ha_Bohr
        * au_to_kJ_conversion
        / 1.889726125
    )

    # ------------------------------------------------------------
    # LJ energy + gradient
    # ------------------------------------------------------------

    E_LJ, grad_LJ = qm_water_lj_energy_gradient(
        coords_A,
        np.array([O]),
        qm_atom_types
    )

    # grad_LJ is already kJ/mol/Angstrom
    # based on the gradient function we previously verified.

    # ------------------------------------------------------------
    # Total energy
    # ------------------------------------------------------------

    E_total = (
        E_qmmm_Ha * au_to_kJ_conversion
        + E_LJ
    )

    # ------------------------------------------------------------
    # Total gradient
    # ------------------------------------------------------------

    grad_total = grad_qmmm + grad_LJ

    return E_total, grad_total.ravel()

def optimize_qmmm_lj_at_distance(
    mol,
    distance,
    water_x,
    maxsteps=100
):
    """
    Stage 1:

    - Fixed water
    - Fixed C at origin
    - Fixed O on the -x axis
    - Fixed C-O distance
    - Optimize all other QM coordinates
    """

    coords0 = mol.atom_coords(unit="Angstrom")

    # ------------------------------------------------------------
    # Fixed atoms
    # ------------------------------------------------------------

    C_index = 2
    O_index = 0

    # ------------------------------------------------------------
    # Initial internal geometry
    # ------------------------------------------------------------

    C0 = coords0[C_index].copy()

    # Translate everything so C is at the origin
    coords0 = coords0 - C0

    # Put O at the requested C-O distance
    coords0[O_index] = np.array([
        -distance,
        0.0,
        0.0
    ])

    coords0[C_index] = np.array([
        0.0,
        0.0,
        0.0
    ])

    # ------------------------------------------------------------
    # Variables
    #
    # We optimize atoms 1, 3, 4, 5, 6.
    #
    # O and C are fixed.
    # ------------------------------------------------------------

    variable_atoms = [1, 3, 4, 5, 6]

    x0 = coords0[variable_atoms].ravel()

    # ------------------------------------------------------------
    # Reconstruct full QM geometry
    # ------------------------------------------------------------

    def make_coords(x):

        coords = coords0.copy()

        coords[variable_atoms] = x.reshape(
            len(variable_atoms), 3
        )

        return coords

    # ------------------------------------------------------------
    # Objective
    # ------------------------------------------------------------

    def objective(x):

        coords = make_coords(x)

        E, grad = qmmm_lj_energy_gradient(
            coords.ravel(),
            water_x
        )

        # Only return gradients for variable atoms
        grad = grad.reshape(-1, 3)

        grad_variable = grad[variable_atoms]

        return E, grad_variable.ravel()

    # ------------------------------------------------------------
    # Optimize
    # ------------------------------------------------------------

    result = minimize(
        fun=lambda x: objective(x)[0],
        x0=x0,
        jac=lambda x: objective(x)[1],
        method="BFGS",
        options={
            "maxiter": maxsteps,
            "gtol": 1e-5,
            "disp": True
        }
    )

    # ------------------------------------------------------------
    # Final geometry
    # ------------------------------------------------------------

    coords_opt = make_coords(result.x)

    mol_opt = build_molecule(coords_opt)

    # ------------------------------------------------------------
    # Final bare QM energy
    # ------------------------------------------------------------

    mf_qm = make_scf(mol_opt)
    energy_qm = mf_qm.kernel()

    # ------------------------------------------------------------
    # Final QM/MM energy
    # ------------------------------------------------------------

    O, H1, H2, M = water_from_variables(water_x)

    mm_coords = np.array([
        H1,
        H2,
        M
    ])

    mm_charges = np.array([
        0.58,
        0.58,
        -1.16
    ])

    mf_qmmm = mm_charge(
        scf.RHF(mol_opt),
        mm_coords,
        mm_charges,
        unit="Angstrom"
    )

    mf_qmmm.conv_tol = SCF_CONV_TOL

    energy_qmmm = mf_qmmm.kernel()

    # ------------------------------------------------------------
    # Final LJ energy
    # ------------------------------------------------------------

    E_LJ = qm_water_lj_energy(
        coords_opt,
        np.array([O]),
        qm_atom_types
    )

    return (
        mol_opt,
        energy_qm,
        energy_qmmm,
        E_LJ,
        result
    )

In [4]:
qm_lj = {
    "OH_O": {
        "sigma_A": 3.400,
        "epsilon_kJmol": 0.2508914038369354
    },

    "OH_H": {
        "sigma_A": 1.443,
        "epsilon_kJmol": 0.18390926154006562
    },

    "C": {
        "sigma_A": 3.39967,
        "epsilon_kJmol": 0.457730
    },

    "Cl": {
        "sigma_A": 4.04468018036,
        "epsilon_kJmol": 0.6276
    },

    "CH3_H": {
        "sigma_A": 2.64953,
        "epsilon_kJmol": 0.0656888
    }
}


In [5]:
qm_atom_types = [
    "OH_O",
    "OH_H",
    "C",
    "Cl",
    "CH3_H",
    "CH3_H",
    "CH3_H"
]

In [6]:
symbols = [
    "O",
    "H",
    "C",
    "Cl",
    "H",
    "H",
    "H"
]

In [7]:
# Fixed TIP4P-D geometry in a local coordinate system
water_O_local = np.array([0.0, 0.0, 0.0])
water_H1_local = np.array([0.9572, 0.0, 0.0])
water_H2_local = np.array([
    -0.23998617,
     0.92662747,
     0.0
])

water_M_local = np.array([
    0.09462759,
    0.12225716,
    0.0
])

In [8]:
# Initial TIP4P-D water configuration
water_x = np.array([
    2.5, 0.0, 3.0,    # O position
    0.0, 0.0, 0.0     # rotation vector
])

In [9]:
# Testing QM optimization
test_mol = build_molecule(
    np.array([
        [-5.000,  0.000,  0.000],
        [-5.970,  0.000,  0.000],
        [ 0.000,  0.000,  0.000],
        [ 1.780,  0.000,  0.000],
        [-0.630,  0.630,  0.630],
        [-0.630, -0.630,  0.630],
        [-0.630,  0.000, -0.890],
    ])
)
test_distance = 3.0
test_mol_opt, test_energy = optimize_at_distance(
    test_mol,
    test_distance,
    maxsteps=25
)

coords_opt = test_mol_opt.atom_coords(
    unit="Angstrom"
)

actual_distance = np.linalg.norm(
    coords_opt[0] - coords_opt[2]
)

print("\nRESULT")
print("Target C-O distance:", test_distance, "Å")
print("Actual C-O distance:", actual_distance, "Å")
print("Final QM energy:", test_energy, "Hartree")


coords_initial = test_mol.atom_coords(
    unit="Angstrom"
)

print("\nInitial coordinates:")
print(coords_initial)

print("\nOptimized coordinates:")
print(coords_opt)

geometric-optimize called with the following command line:
/home/chemistry/venvs/jupyter/lib/python3.14/site-packages/ipykernel_launcher.py -f /home/chemistry/.local/share/jupyter/runtime/kernel-caa6aab5-dbc5-4081-be18-bb2bff3b24f0.json

                                        ())))))))))))))))/                     
                                    ())))))))))))))))))))))))),                
                                *)))))))))))))))))))))))))))))))))             
                        #,    ()))))))))/                .)))))))))),          
                      #%%%%,  ())))))                        .))))))))*        
                      *%%%%%%,  ))              ..              ,))))))).      
                        *%%%%%%,         ***************/.        .)))))))     
                #%%/      (%%%%%%,    /*********************.       )))))))    
              .%%%%%%#      *%%%%%%,  *******/,     **********,      .))))))   
                .%%%%%%/      *%%%%%%,  **


Geometry optimization cycle 1
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   O  -5.000000   0.000000   0.000000    0.000000  0.000000  0.000000
   H  -5.970000   0.000000   0.000000    0.000000  0.000000  0.000000
   C   0.000000   0.000000   0.000000    0.000000  0.000000  0.000000
  Cl   1.780000   0.000000   0.000000    0.000000  0.000000  0.000000
   H  -0.630000   0.630000   0.630000    0.000000  0.000000  0.000000
   H  -0.630000  -0.630000   0.630000    0.000000  0.000000  0.000000
   H  -0.630000   0.000000  -0.890000    0.000000  0.000000  0.000000

WARN: Mole.unit (Angstrom) is changed to Bohr

converged SCF energy = -574.308632234647
--------------- RHF_Scanner gradients ---------------
         x                y                z
0 O    -0.0100830307     0.0000000000    -0.0003427109
1 H     0.0073938184    -0.0000000000     0.0000113669
2 C     0.1146414712    -0.0000000000    -0.0269586810
3 Cl     0.0056602441     0.0

Step    0 : Gradient = 5.811e-02/1.004e-01 (rms/max) Energy = -574.3086322346
Hessian Eigenvalues: 5.00000e-02 5.00000e-02 5.00000e-02 ... 3.46752e-01 3.47392e-01 5.01282e-01



Geometry optimization cycle 2
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   O  -4.274293  -0.000000   0.000000    0.725707 -0.000000  0.000000
   H  -5.534537   0.000000  -0.000000    0.435463  0.000000 -0.000000
   C  -0.693718   0.000000   0.000296   -0.693718  0.000000  0.000296
  Cl   1.202359  -0.000000  -0.046073   -0.577641 -0.000000 -0.046073
   H  -1.153006   0.646901   0.647624   -0.523006  0.016901  0.017624
   H  -1.153006  -0.646901   0.647624   -0.523006 -0.016901  0.017624
   H  -1.215552   0.000000  -0.879472   -0.585552  0.000000  0.010528
converged SCF energy = -574.30697958984
--------------- RHF_Scanner gradients ---------------
         x                y                z
0 O     0.0842473412    -0.0000000000    -0.0005904186
1 H    -0.0921802783     0.0000000000    -0.0001401470
2 C    -0.0064944829     0.0000000000    -0.0238242789
3 Cl     0.0300165832    -0.0000000000    -0.0011177672
4 H    -0.0086637070  

Step    1 : Displace = 5.333e-01/9.746e-01 (rms/max) Trust = 1.000e-01 (=) Grad_T = 6.910e-02/9.218e-02 (rms/max) E (change) = -574.3069795898 (+1.653e-03) Quality = 0.003
Constraint                         Current      Target       Diff.
Distance 1-3                       3.58057     3.00000     0.58057
Hessian Eigenvalues: 4.37516e-03 5.00000e-02 5.00000e-02 ... 3.47323e-01 3.98539e-01 5.71866e-01


...


Geometry optimization cycle 25
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   O  -4.094338  -0.000000  -0.599503   -0.001516 -0.000000  0.007136
   H  -4.688585   0.000000   0.168030   -0.005068  0.000000  0.004556
   C  -1.274528   0.000000   0.416339    0.004358  0.000000 -0.005857
  Cl   0.166942   0.000000  -0.852092   -0.002129  0.000000 -0.011227
   H  -1.104666   0.890372   0.996617    0.006624 -0.000486 -0.005836
   H  -1.104666  -0.890372   0.996617    0.006624  0.000486 -0.005836
   H  -2.260772  -0.000000  -0.108799    0.000983  0.000000 -0.000477
converged SCF energy = -574.40839545236
--------------- RHF_Scanner gradients ---------------
         x                y                z
0 O    -0.0086189143     0.0000000000    -0.0015298301
1 H    -0.0007707228    -0.0000000000    -0.0010871402
2 C     0.0083580138     0.0000000000     0.0012780306
3 Cl     0.0003522949    -0.0000000000     0.0005071529
4 H     0.0002877284 

Step   24 : Displace = 3.147e-03/7.197e-03 (rms/max) Trust = 3.146e-03 (-) Grad_T = 1.862e-04/3.442e-04 (rms/max) E (change) = -574.4083954524 (+4.088e-05) Quality = 0.964
Constraint                         Current      Target       Diff.
Distance 1-3                       2.99721     3.00000    -0.00279
Hessian Eigenvalues: 4.30029e-03 1.33523e-02 2.35412e-02 ... 3.73762e-01 3.98871e-01 5.00142e-01



Geometry optimization cycle 26
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   O  -4.096367  -0.000000  -0.587201   -0.002029 -0.000000  0.012301
   H  -4.694442   0.000000   0.177470   -0.005857  0.000000  0.009440
   C  -1.269006   0.000000   0.406662    0.005522 -0.000000 -0.009677
  Cl   0.158925   0.000000  -0.875727   -0.008017  0.000000 -0.023635
   H  -1.093611   0.889813   0.985953    0.011055 -0.000559 -0.010664
   H  -1.093611  -0.889813   0.985953    0.011055  0.000559 -0.010664
   H  -2.260674  -0.000000  -0.108512    0.000098 -0.000000  0.000287
converged SCF energy = -574.408373823935
--------------- RHF_Scanner gradients ---------------
         x                y                z
0 O    -0.0085031505     0.0000000000    -0.0015370714
1 H    -0.0009354191    -0.0000000000    -0.0009959554
2 C     0.0085998560     0.0000000000     0.0010763270
3 Cl     0.0002470561    -0.0000000000     0.0005710614
4 H     0.0002739131

Step   25 : Displace = 4.559e-03/9.464e-03 (rms/max) Trust = 4.449e-03 (+) Grad_T = 2.636e-04/4.754e-04 (rms/max) E (change) = -574.4083738239 (+2.163e-05) Quality = 0.899
Constraint                         Current      Target       Diff.
Distance 1-3                       2.99695     3.00000    -0.00305
Hessian Eigenvalues: 4.30029e-03 1.33523e-02 2.35412e-02 ... 3.73762e-01 3.98871e-01 5.00142e-01
Maximum iterations reached (25); increase --maxiter for more


Geometry optimization failed to converge in 25 iterations
converged SCF energy = -574.408373824827

RESULT
Target C-O distance: 3.0 Å
Actual C-O distance: 2.996954854898371 Å
Final QM energy: -574.408373824827 Hartree

Initial coordinates:
[[-5.    0.    0.  ]
 [-5.97  0.    0.  ]
 [ 0.    0.    0.  ]
 [ 1.78  0.    0.  ]
 [-0.63  0.63  0.63]
 [-0.63 -0.63  0.63]
 [-0.63  0.   -0.89]]

Optimized coordinates:
[[-4.09636716e+00 -5.31901347e-12 -5.87201439e-01]
 [-4.69444235e+00  5.75377106e-12  1.77470015e-01]
 [-1.26900563e+00  9.76901054e-13  4.06662308e-01]
 [ 1.58925157e-01  1.22344055e-12 -8.75726834e-01]
 [-1.09361104e+00  8.89812825e-01  9.85952742e-01]
 [-1.09361104e+00 -8.89812825e-01  9.85952742e-01]
 [-2.26067365e+00 -9.01746196e-13 -1.08512397e-01]]


In [11]:
# Testing electrostatic-only QM/MM optimization

test_water_x = np.array([
    2.5, 0.0, 3.0,
    0.0, 0.0, 0.0
])

test_distance = 3.0

mol_qmmm, E_qm, E_qmmm = optimize_qmmm_at_distance(
    test_mol,
    test_distance,
    test_water_x,
    maxiter=25
)

print("\nRESULT")
print("Bare QM energy:")
print(E_qm, "Hartree")

print("\nQM/MM energy:")
print(E_qmmm, "Hartree")

E_electrostatic = (
    E_qmmm - E_qm
) * au_to_kJ_conversion

print("\nElectrostatic interaction:")
print(E_electrostatic, "kJ/mol")

coords_final = mol_qmmm.atom_coords(unit="Angstrom")

actual_distance = np.linalg.norm(
    coords_final[0] - coords_final[2]
)

print("\nTarget C-O distance:")
print(test_distance, "Å")

print("Actual C-O distance:")
print(actual_distance, "Å")

print("\nOptimized coordinates:")
print(coords_final)

converged SCF energy = -574.305014520736
--------------- QMMMRHF gradients ---------------
         x                y                z
0 O    -0.0099856768    -0.0002517758    -0.0002165533
1 H     0.0075974117     0.0000311981    -0.0000022025
2 C     0.1137643966     0.0001287242    -0.0268861286
3 Cl     0.0061025534    -0.0007342618    -0.0000192679
4 H    -0.0406633867    -0.0645841297     0.0069044847
5 H    -0.0413168782     0.0647130735     0.0073871398
6 H    -0.0356751281     0.0000087939     0.0142266191
----------------------------------------------
converged SCF energy = -574.082851396975
--------------- QMMMRHF gradients ---------------
         x                y                z
0 O     0.0418749018    -0.0001482973    -0.0002002182
1 H    -0.0618089289    -0.0000563558    -0.0001583509
2 C    -0.0498183995     0.0000772278    -0.0118823421
3 Cl     0.0157814625    -0.0024563492     0.0028082103
4 H     0.0174428821    -0.1140048632    -0.0373474703
5 H     0.017082205

ValueError: too many values to unpack (expected 3, got 4)

In [ ]:
# Make sure the water position actually matters
coords = mol_qmmm.atom_coords(unit="Angstrom")

test_mol = build_molecule(coords)

water_tests = [
    np.array([2.5, 0.0, 3.0, 0.0, 0.0, 0.0]),
    np.array([2.5, 0.0, 4.0, 0.0, 0.0, 0.0]),
    np.array([3.5, 0.0, 3.0, 0.0, 0.0, 0.0]),
]

for wx in water_tests:

    O, H1, H2, M = water_from_variables(wx)

    mm_coords = np.array([H1, H2, M])
    mm_charges = np.array([0.58, 0.58, -1.16])

    mf_qm = make_scf(test_mol)
    E_qm = mf_qm.kernel()

    mf_qmmm = mm_charge(
        scf.RHF(test_mol),
        mm_coords,
        mm_charges,
        unit="Angstrom"
    )
    mf_qmmm.conv_tol = SCF_CONV_TOL
    E_qmmm = mf_qmmm.kernel()

    E_elec = (
        E_qmmm - E_qm
    ) * au_to_kJ_conversion

    print(
        f"Water O = {O}, "
        f"E_elec = {E_elec:.6f} kJ/mol"
    )

In [ ]:
# Verifying that the LJ energy behaves sensibly for these same three water positions.
for wx in water_tests:

    O, H1, H2, M = water_from_variables(wx)

    E_LJ = qm_water_lj_energy(
        test_mol.atom_coords(unit="Angstrom"),
        np.array([O]),
        qm_atom_types
    )

    print(
        f"Water O = {O}, "
        f"E_LJ = {E_LJ:.6f} kJ/mol"
    )

In [ ]:
for wx in water_tests:

    O, H1, H2, M = water_from_variables(wx)

    mm_coords = np.array([H1, H2, M])
    mm_charges = np.array([0.58, 0.58, -1.16])

    # Bare QM
    mf_qm = make_scf(test_mol)
    E_qm = mf_qm.kernel()

    # QM/MM electrostatics
    mf_qmmm = mm_charge(
        scf.RHF(test_mol),
        mm_coords,
        mm_charges,
        unit="Angstrom"
    )
    mf_qmmm.conv_tol = SCF_CONV_TOL
    E_qmmm = mf_qmmm.kernel()

    E_elec = (
        E_qmmm - E_qm
    ) * au_to_kJ_conversion

    # LJ
    E_LJ = qm_water_lj_energy(
        test_mol.atom_coords(unit="Angstrom"),
        np.array([O]),
        qm_atom_types
    )

    print("\nWater O:", O)
    print(f"Electrostatic: {E_elec:.6f} kJ/mol")
    print(f"LJ:            {E_LJ:.6f} kJ/mol")
    print(f"Total:         {E_elec + E_LJ:.6f} kJ/mol")

In [ ]:
# Testing the gradient function
qm_coords_A = test_mol.atom_coords(unit="Angstrom")

water_O = np.array([2.5, 0.0, 3.0])

E_old = qm_water_lj_energy(
    qm_coords_A,
    np.array([water_O]),
    qm_atom_types
)

E_new, grad = qm_water_lj_energy_gradient(
    qm_coords_A,
    np.array([water_O]),
    qm_atom_types
)

print("Old LJ energy:", E_old)
print("New LJ energy:", E_new)
print("Difference:", E_new - E_old)

print("\nLJ gradient (kJ/mol/Angstrom):")
print(grad)

In [ ]:
# checks that the gradient is actually the derivative of the LJ energy.
delta = 1e-5

i = 0       # QM atom
j = 0       # x coordinate

coords_plus = qm_coords_A.copy()
coords_minus = qm_coords_A.copy()

coords_plus[i, j] += delta
coords_minus[i, j] -= delta

E_plus = qm_water_lj_energy(
    coords_plus,
    np.array([water_O]),
    qm_atom_types
)

E_minus = qm_water_lj_energy(
    coords_minus,
    np.array([water_O]),
    qm_atom_types
)

numerical_gradient = (
    E_plus - E_minus
) / (2 * delta)

print("Analytic gradient:",
      grad[i, j])

print("Numerical gradient:",
      numerical_gradient)

print("Difference:",
      grad[i, j] - numerical_gradient)

In [ ]:
# Testing whether QM/MM optimization works

# ------------------------------------------------
# Test optimize_qmmm_at_distance()
# ------------------------------------------------

test_mol = build_molecule(
    np.array([
        [-5.000,  0.000,  0.000],
        [-5.970,  0.000,  0.000],
        [ 0.000,  0.000,  0.000],
        [ 1.780,  0.000,  0.000],
        [-0.630,  0.630,  0.630],
        [-0.630, -0.630,  0.630],
        [-0.630,  0.000, -0.890],
    ])
)

target_distance = 3.0

# Save initial coordinates
initial_coords = test_mol.atom_coords(
    unit="Angstrom"
).copy()

# Initial bare QM energy
mf_qm_initial = make_scf(test_mol)
E_qm_initial = mf_qm_initial.kernel()

print("\nInitial bare QM energy:")
print(E_qm_initial, "Hartree")

# ------------------------------------------------
# Call the actual function
# ------------------------------------------------

optimized_mol, E_qmmm = optimize_qmmm_at_distance(
    test_mol,
    target_distance,
    water_x,
    maxsteps=25
)

# ------------------------------------------------
# Final bare QM energy
# ------------------------------------------------

mf_qm_final = make_scf(optimized_mol)
E_qm_final = mf_qm_final.kernel()

# ------------------------------------------------
# Final coordinates
# ------------------------------------------------

final_coords = optimized_mol.atom_coords(
    unit="Angstrom"
)

# Actual C-O distance
actual_distance = np.linalg.norm(
    final_coords[0] - final_coords[2]
)

# ------------------------------------------------
# Report
# ------------------------------------------------

print("\n" + "=" * 60)
print("RESULT")
print("=" * 60)

print(f"Target C-O distance: {target_distance:.6f} Å")
print(f"Actual C-O distance: {actual_distance:.8f} Å")

print("\nInitial bare QM energy:")
print(f"{E_qm_initial:.12f} Hartree")

print("\nFinal bare QM energy:")
print(f"{E_qm_final:.12f} Hartree")

print("\nFinal QM/MM energy:")
print(f"{E_qmmm:.12f} Hartree")

print("\nQM/MM - bare QM:")
print(
    f"{(E_qmmm - E_qm_final) * au_to_kJ_conversion:.6f} kJ/mol"
)

print("\nInitial coordinates:")
print(initial_coords)

print("\nOptimized coordinates:")
print(final_coords)

In [ ]:
# Testing the gradient
# Build the current TIP4P-D water
O, H1, H2, M = water_from_variables(water_x)

mm_coords = np.array([
    H1,
    H2,
    M
])

mm_charges = np.array([
    0.58,
    0.58,
    -1.16
])

# QM/MM SCF
mf_qmmm = mm_charge(
    scf.RHF(test_mol),
    mm_coords,
    mm_charges,
    unit="Angstrom"
)

mf_qmmm.conv_tol = SCF_CONV_TOL

mf_qmmm.kernel()

# Build combined gradient
grad = QMMM_LJ_Gradients(
    mf_qmmm,
    O,
    qm_atom_types
)

# Calculate combined gradient
combined_gradient = grad.kernel()

print("Combined gradient:")
print(combined_gradient)

print("\nGradient shape:")
print(combined_gradient.shape)

print("\nMaximum absolute gradient:")
print(np.max(np.abs(combined_gradient)))

In [ ]:
# Re-running the gradient test
O, H1, H2, M = water_from_variables(water_x)

mm_coords = np.array([
    H1,
    H2,
    M
])

mm_charges = np.array([
    0.58,
    0.58,
    -1.16
])

mf_qmmm = mm_charge(
    scf.RHF(test_mol),
    mm_coords,
    mm_charges,
    unit="Angstrom"
)

mf_qmmm.conv_tol = SCF_CONV_TOL
mf_qmmm.kernel()

grad = QMMM_LJ_Gradients(
    mf_qmmm,
    O,
    qm_atom_types
)

combined_gradient = grad.kernel()

print("Combined gradient:")
print(combined_gradient)

print("\nShape:")
print(combined_gradient.shape)

print("\nMax gradient:")
print(np.max(np.abs(combined_gradient)))

In [ ]:
# # Testing optimize_qmmm_lj_at_distance at one frame (this failed)
# test_mol = build_molecule(
#     np.array([
#         [-5.000,  0.000,  0.000],
#         [-5.970,  0.000,  0.000],
#         [ 0.000,  0.000,  0.000],
#         [ 1.780,  0.000,  0.000],
#         [-0.630,  0.630,  0.630],
#         [-0.630, -0.630,  0.630],
#         [-0.630,  0.000, -0.890],
#     ])
# )

# optimized_mol, energy_qm, energy_qmmm, E_LJ = \
#     optimize_qmmm_lj_at_distance(
#         test_mol,
#         3.0,
#         water_x,
#         maxsteps=25
#     )

# coords = optimized_mol.atom_coords(
#     unit="Angstrom"
# )

# actual_distance = np.linalg.norm(
#     coords[0] - coords[2]
# )

# print("\n" + "=" * 60)
# print("QM/MM + LJ OPTIMIZATION")
# print("=" * 60)

# print(f"Target C-O: {3.0:.6f} Å")
# print(f"Actual C-O: {actual_distance:.8f} Å")

# print(f"\nBare QM:    {energy_qm:.12f} Hartree")
# print(f"QM/MM:      {energy_qmmm:.12f} Hartree")
# print(f"LJ:         {E_LJ:.6f} kJ/mol")

# print(
#     "\nElectrostatic interaction:",
#     (energy_qmmm - energy_qm)
#     * au_to_kJ_conversion,
#     "kJ/mol"
# )

In [ ]:
test_mol = build_molecule(
    np.array([
        [-5.000,  0.000,  0.000],
        [-5.970,  0.000,  0.000],
        [ 0.000,  0.000,  0.000],
        [ 1.780,  0.000,  0.000],
        [-0.630,  0.630,  0.630],
        [-0.630, -0.630,  0.630],
        [-0.630,  0.000, -0.890],
    ])
)

E_qmmm, grad_qmmm = qmmm_gradient(
    test_mol,
    water_x
)

print("QM/MM energy:")
print(E_qmmm, "Hartree")

print("\nGradient:")
print(grad_qmmm)

print("\nMaximum gradient:")
print(np.max(np.abs(grad_qmmm)))

In [ ]:
# coords0 = test_mol.atom_coords(unit="Angstrom")

# result = minimize(
#     lambda x: qmmm_objective(x, water_x),
#     coords0.reshape(-1),
#     jac=True,
#     method="BFGS"
# )

# print(result)

In [ ]:
optimized_mol, energy_qm, energy_qmmm, result = \
    optimize_qmmm_at_distance(
        test_mol,
        3.0,
        water_x,
        maxiter=25
    )

coords_opt = optimized_mol.atom_coords(
    unit="Angstrom"
)

actual_distance = np.linalg.norm(
    coords_opt[0] - coords_opt[2]
)

print("\nRESULT")
print("Optimization success:", result.success)
print("Message:", result.message)

print("\nTarget C-O distance:")
print(3.0, "Å")

print("\nActual C-O distance:")
print(actual_distance, "Å")

print("\nBare QM energy:")
print(energy_qm, "Hartree")

print("\nQM/MM energy:")
print(energy_qmmm, "Hartree")

print("\nQM/MM - bare QM:")
print(
    (energy_qmmm - energy_qm) * au_to_kJ_conversion,
    "kJ/mol"
)

In [ ]:
# test qmmm_lj_objective
E_test, grad_test = qmmm_lj_objective(
    test_mol.atom_coords(unit="Angstrom").reshape(-1),
    water_x
)

print("QM/MM + LJ energy:")
print(E_test, "Hartree")

print("\nGradient shape:")
print(grad_test.shape)

print("\nMaximum gradient:")
print(np.max(np.abs(grad_test)))


coords_A = test_mol.atom_coords(unit="Angstrom")

E_qmmm, grad_qmmm = qmmm_gradient(
    test_mol,
    water_x
)

O, H1, H2, M = water_from_variables(water_x)

E_LJ, grad_LJ = qm_water_lj_energy_gradient(
    coords_A,
    np.array([O]),
    qm_atom_types
)

print("QM/MM:")
print(E_qmmm, "Hartree")

print("LJ:")
print(E_LJ, "kJ/mol")

print("LJ:")
print(E_LJ / au_to_kJ_conversion, "Hartree")

print("Total:")
print(E_qmmm + E_LJ / au_to_kJ_conversion, "Hartree")

In [ ]:
print("Testing QMMM_LJ_Gradients interface...")

O, H1, H2, M = water_from_variables(water_x)

mm_coords = np.array([
    H1,
    H2,
    M
])

mm_charges = np.array([
    0.58,
    0.58,
    -1.16
])

mf = mm_charge(
    scf.RHF(test_mol),
    mm_coords,
    mm_charges,
    unit="Angstrom"
)

mf.conv_tol = SCF_CONV_TOL

grad = QMMM_LJ_Gradients(
    mf,
    O,
    qm_atom_types,
    mm_coords,
    mm_charges
)

print("nuc_grad_method:", grad.nuc_grad_method())
print("as_scanner:", grad.as_scanner())
print("converged:", grad.converged)

E, G = grad(test_mol)

print("\nEnergy:", E)
print("Gradient shape:", G.shape)
print("Maximum gradient:", np.max(np.abs(G)))

In [ ]:
# Optimizing the QM geometry with fixed water and lj
optimized_mol, energy_qm, energy_qmmm, E_LJ, result = \
    optimize_qmmm_lj_at_distance(
        test_mol,
        3.0,
        water_x,
        maxsteps=100
    )

In [ ]:
# ------------------------------------------------------------
# Verify TOTAL QM/MM + LJ gradient numerically
# ------------------------------------------------------------

O, H1, H2, M = water_from_variables(water_x)

mm_coords = np.array([
    H1,
    H2,
    M
])

mm_charges = np.array([
    0.58,
    0.58,
    -1.16
])

# Build the gradient object
mf = mm_charge(
    scf.RHF(test_mol),
    mm_coords,
    mm_charges,
    unit="Angstrom"
)

mf.conv_tol = SCF_CONV_TOL

grad_object = QMMM_LJ_Gradients(
    mf,
    O,
    qm_atom_types,
    mm_coords,
    mm_charges
)

# Analytic energy and gradient
E0, G0 = grad_object(test_mol)

print("Analytic total energy:")
print(E0)

print("\nAnalytic gradient:")
print(G0)

# ------------------------------------------------------------
# Numerical derivative for one coordinate
# ------------------------------------------------------------

atom = 0
coord = 0
delta_A = 1.0e-4

coords_plus = test_mol.atom_coords(unit="Angstrom")
coords_minus = test_mol.atom_coords(unit="Angstrom")

coords_plus[atom, coord] += delta_A
coords_minus[atom, coord] -= delta_A

mol_plus = build_molecule(coords_plus)
mol_minus = build_molecule(coords_minus)

E_plus, _ = grad_object(mol_plus)
E_minus, _ = grad_object(mol_minus)

# Energy is Hartree, displacement is Angstrom
numerical = (
    E_plus - E_minus
) / (2.0 * delta_A)

# Convert Hartree/Angstrom -> Hartree/Bohr
numerical_Ha_Bohr = numerical * 1.889726125

print("\nGradient check for atom", atom, "coordinate", coord)
print("Analytic: ", G0[atom, coord])
print("Numerical: ", numerical_Ha_Bohr)
print("Difference:",
      G0[atom, coord] - numerical_Ha_Bohr)

In [ ]:
# ------------------------------------------------------------
# Separate QM/MM and LJ gradient checks
# ------------------------------------------------------------

O, H1, H2, M = water_from_variables(water_x)

mm_coords = np.array([
    H1,
    H2,
    M
])

mm_charges = np.array([
    0.58,
    0.58,
    -1.16
])

# QM/MM object
mf_qmmm = mm_charge(
    scf.RHF(test_mol),
    mm_coords,
    mm_charges,
    unit="Angstrom"
)

mf_qmmm.conv_tol = SCF_CONV_TOL

E_qmmm = mf_qmmm.kernel()

grad_qmmm = mf_qmmm.nuc_grad_method().kernel()

# LJ
coords_A = test_mol.atom_coords(unit="Angstrom")

E_LJ, grad_LJ = qm_water_lj_energy_gradient(
    coords_A,
    O,
    qm_atom_types
)

grad_LJ_Ha_Bohr = (
    grad_LJ
    / au_to_kJ_conversion
    / 1.889726125
)

print("QM/MM gradient:")
print(grad_qmmm[0])

print("\nLJ gradient:")
print(grad_LJ_Ha_Bohr[0])

print("\nCombined analytic gradient:")
print((grad_qmmm + grad_LJ_Ha_Bohr)[0])

In [ ]:
optimized_mol, energy_qm, energy_qmmm, E_LJ, result = \
    optimize_qmmm_lj_at_distance(
        test_mol,
        3.0,
        water_x,
        maxsteps=50
    )

coords_opt = optimized_mol.atom_coords(unit="Angstrom")

actual_distance = np.linalg.norm(
    coords_opt[0] - coords_opt[2]
)

print("\nRESULT")
print("Optimization success:", result.success)
print("Message:", result.message)

print("\nTarget C-O distance:")
print(f"{3.0:.8f} Å")

print("\nActual C-O distance:")
print(f"{actual_distance:.8f} Å")

print("\nBare QM energy:")
print(f"{energy_qm:.12f} Hartree")

print("\nQM/MM energy:")
print(f"{energy_qmmm:.12f} Hartree")

print("\nQM/MM - bare QM:")
print(
    (energy_qmmm - energy_qm)
    * au_to_kJ_conversion,
    "kJ/mol"
)

print("\nLJ energy:")
print(f"{E_LJ:.8f} kJ/mol")

print("\nCoordinates:")
print(coords_opt)

In [ ]:
# ============================================================
# STAGE 2: Move one fixed water molecule around a fixed QM system
# ============================================================

coords_A = test_mol.atom_coords(unit="Angstrom")

# Keep the water orientation fixed.
# We will vary only its x-coordinate.
x_positions = np.array([1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5])

results = []

for x in x_positions:

    water_test = water_x.copy()
    water_test[0] = x

    E_total = total_qmmm_lj_energy(
        coords_A,
        water_test
    )

    # Also calculate the two components separately
    O, H1, H2, M = water_from_variables(water_test)

    mm_coords = np.array([H1, H2, M])
    mm_charges = np.array([0.58, 0.58, -1.16])

    mf_qmmm = mm_charge(
        scf.RHF(test_mol),
        mm_coords,
        mm_charges,
        unit="Angstrom"
    )

    mf_qmmm.conv_tol = SCF_CONV_TOL
    E_qmmm = mf_qmmm.kernel()

    E_qm = make_scf(test_mol).kernel()

    E_electrostatic = (
        E_qmmm - E_qm
    ) * au_to_kJ_conversion

    E_LJ = qm_water_lj_energy(
        coords_A,
        np.array([O]),
        qm_atom_types
    )

    results.append([
        x,
        E_electrostatic,
        E_LJ,
        E_electrostatic + E_LJ
    ])

print("\nWater position and interaction energies")
print("x (Å)    Electrostatic    LJ          Total")
print("------------------------------------------------")

for row in results:
    print(
        f"{row[0]:5.2f}    "
        f"{row[1]:12.6f}    "
        f"{row[2]:10.6f}    "
        f"{row[3]:10.6f}"
    )

In [ ]:
# ============================================================
# STAGE 2B: Numerical derivative of the water-position energy
# ============================================================

def water_total_energy_at_x(x):

    water_test = water_x.copy()
    water_test[0] = x

    return total_qmmm_lj_energy(
        coords_A,
        water_test
    )


delta = 0.001  # Angstrom

test_positions = [2.5, 3.0, 3.5, 4.0]

print("Numerical dE/dx")
print("-----------------------------")

for x in test_positions:

    E_minus = water_total_energy_at_x(x - delta)
    E_plus  = water_total_energy_at_x(x + delta)

    dE_dx = (E_plus - E_minus) / (2 * delta)

    print(
        f"x = {x:.2f} Å   "
        f"dE/dx = {dE_dx:.8f} kJ/mol/Å"
    )


In [ ]:
# ============================================================
# STAGE 2C: Check electrostatic and LJ derivatives separately
# ============================================================

def water_components_at_x(x):

    water_test = water_x.copy()
    water_test[0] = x

    O, H1, H2, M = water_from_variables(water_test)

    # ----------------------------
    # QM/MM electrostatics
    # ----------------------------

    mm_coords = np.array([
        H1,
        H2,
        M
    ])

    mm_charges = np.array([
        0.58,
        0.58,
        -1.16
    ])

    mf_qmmm = mm_charge(
        scf.RHF(test_mol),
        mm_coords,
        mm_charges,
        unit="Angstrom"
    )

    mf_qmmm.conv_tol = SCF_CONV_TOL
    E_qmmm = mf_qmmm.kernel()

    E_qm = make_scf(test_mol).kernel()

    E_electrostatic = (
        E_qmmm - E_qm
    ) * au_to_kJ_conversion

    # ----------------------------
    # LJ
    # ----------------------------

    E_LJ = qm_water_lj_energy(
        coords_A,
        np.array([O]),
        qm_atom_types
    )

    return E_electrostatic, E_LJ


delta = 0.001

test_positions = [2.5, 3.0, 3.5, 4.0]

print("Derivative check")
print()
print(
    "x       dE_elec/dx     dE_LJ/dx       "
    "sum            dE_total/dx"
)
print("-" * 75)

for x in test_positions:

    Eelec_minus, ELJ_minus = water_components_at_x(x - delta)
    Eelec_plus,  ELJ_plus  = water_components_at_x(x + delta)

    dE_elec = (
        Eelec_plus - Eelec_minus
    ) / (2 * delta)

    dE_LJ = (
        ELJ_plus - ELJ_minus
    ) / (2 * delta)

    dE_total = dE_elec + dE_LJ

    # Independent total-energy derivative
    E_total_minus = (
        Eelec_minus + ELJ_minus
    )

    E_total_plus = (
        Eelec_plus + ELJ_plus
    )

    dE_total_direct = (
        E_total_plus - E_total_minus
    ) / (2 * delta)

    print(
        f"{x:3.1f}   "
        f"{dE_elec:14.6f} "
        f"{dE_LJ:14.6f} "
        f"{dE_total:14.6f} "
        f"{dE_total_direct:14.6f}"
    )

In [ ]:
from scipy.optimize import minimize_scalar

def water_x_energy(x_position):

    x = water_x.copy()
    x[0] = x_position

    return total_qmmm_lj_energy(
        test_mol.atom_coords(unit="Angstrom"),
        x
    )


result = minimize_scalar(
    water_x_energy,
    bounds=(2.5, 4.5),
    method="bounded",
    options={"xatol": 1e-4}
)

print(result)
print()
print("Optimal water x:", result.x)
print("Minimum energy:", result.fun, "kJ/mol")

In [ ]:
print("Direct energy:")
print(
    total_qmmm_lj_energy(
        test_mol.atom_coords(unit="Angstrom"),
        np.array([4.5, 0.0, 3.0, 0.0, 0.0, 0.0])
    )
)

print()
print("Energy through water_x_energy:")
print(
    water_x_energy(4.5)
)for x in [2.5, 3.0, 3.5, 4.0, 4.5]:

    water = water_x.copy()
    water[0] = x

    E = total_qmmm_lj_energy(
        test_mol.atom_coords(unit="Angstrom"),
        water
    )

    print(f"x = {x:.2f} Å   E = {E:.6f} kJ/mol")

In [ ]:
for x in [2.5, 3.0, 3.5, 4.0, 4.5, 6, 7, 8]:

    water = water_x.copy()
    water[0] = x

    E = total_qmmm_lj_energy(
        test_mol.atom_coords(unit="Angstrom"),
        water
    )

    print(f"x = {x:.2f} Å   E = {E:.6f} kJ/mol")